In [0]:
from pyspark.sql import SparkSession
from datetime import datetime
import pytz
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime
import pytz


In [0]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
#202410
data_exec_inicial = 202503

# converte YYYYMM -> date
data_dt = datetime.strptime(str(data_exec_inicial), "%Y%m")

# subtrai 12 meses
data_exec_final = int((data_dt - relativedelta(months=5)).strftime("%Y%m"))
data_exec_final

In [0]:
path01 = "hackathon2025.silver.base_score_bureau_movel"
path02 = "hackathon2025.silver.book_pagamento"
path03 = "hackathon2025.silver.book_atraso"

In [0]:
df_atraso = (
    spark.read
         .table(path03)
         .filter(
             (col("SAFRA") >= "202410") &
             (col("SAFRA") <= "202503")
           )
)
df_atraso.createOrReplaceTempView("df_atraso")

df_atraso = df_atraso.drop("rn")


df_atraso.createOrReplaceTempView("df_atraso")

df_atraso.count()

In [0]:
for col in df_atraso.columns:
    agg_result = df_atraso.agg(
        {col: "count"} 
    ).collect()[0]
    
    total = df_atraso.count()
    nao_nulos = agg_result[f"count({col})"]
    nulos = total - nao_nulos
    
    if nulos > 0:
        print(f"{col}: {nulos} nulos ({nulos/total*100:.2f}%)")

In [0]:
display(df_atraso)

In [0]:
df_bureau = (
    spark.read
         .table(path01)
         .filter(
             (col("SAFRA") >= data_exec_final) &
             (col("SAFRA") <= data_exec_inicial)
         )
)
df_bureau.count()

In [0]:
df_temp_01 = (
    df_bureau.alias("b")
    .join(
        df_atraso.alias("a"),
        on=["NUM_CPF", "SAFRA"],
        how="left"
    )
    .drop("rn", "DATPROC")
)

df_temp_01.createOrReplaceTempView("df_temp_01")

df_temp_01.count()


In [0]:
total_rows = df_temp_01.count()
pct_nulos = (
    df_temp_01
    .select(
        (
            F.sum(
                F.when(
                    F.col("VL_MED_U9M_PDD_N_VAL_MULTA_CANCELAMENTO_ATRASO").isNull(), 1
                ).otherwise(0)
            ) * 100 / total_rows
        ).alias("pct_nulos")
    )
)

pct_nulos.show(truncate=False)


In [0]:
display(df_temp_01)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from typing import List, Tuple


def remove_high_missing_columns_sampled(
    df: DataFrame, 
    threshold: float = 70.0,
    sample_fraction: float = 0.1,
    safra_col: str = "SAFRA"
) -> Tuple[DataFrame, List[str]]:
    """
    Remove colunas com percentual de nulos >= threshold usando
    amostragem estratificada por SAFRA (Serverless-safe).
    """

    print(f"Amostragem estratificada por '{safra_col}' ({sample_fraction*100:.1f}%)")

    # ----------------------------------------------------------
    # 1. Coletar safras distintas (SEM RDD)
    # ----------------------------------------------------------
    safras = [
        row[safra_col]
        for row in df.select(safra_col).distinct().collect()
    ]

    # Fração igual para todas as safras
    fractions = {safra: sample_fraction for safra in safras}

    # ----------------------------------------------------------
    # 2. Amostragem estratificada
    # ----------------------------------------------------------
    sampled_df = (
        df
        .sampleBy(
            col=safra_col,
            fractions=fractions,
            seed=42
        )
    )

    total_rows = sampled_df.count()

    print(f"Tamanho da amostra: {total_rows:,} linhas")
    print(f"Total de colunas: {len(df.columns):,}")

    if total_rows == 0:
        print("Amostra vazia — nenhum cálculo realizado.")
        return df, []

    # ----------------------------------------------------------
    # 3. Cálculo de nulos por batch (otimizado)
    # ----------------------------------------------------------
    columns = df.columns
    batch_size = 1000

    columns_to_remove = []
    null_percentages = {}

    for i in range(0, len(columns), batch_size):
        batch_cols = columns[i:i + batch_size]

        # Conta valores NÃO nulos (mais eficiente)
        agg_exprs = [F.count(F.col(c)).alias(c) for c in batch_cols]

        batch_result = sampled_df.agg(*agg_exprs).first()

        for col in batch_cols:
            non_null_count = batch_result[col]
            null_percent = 100 * (1 - non_null_count / total_rows)

            null_percentages[col] = null_percent

            if null_percent >= threshold:
                columns_to_remove.append(col)

        print(
            f"Processado lote {i // batch_size + 1}/"
            f"{(len(columns) + batch_size - 1) // batch_size}"
        )

    # ----------------------------------------------------------
    # 4. Drop final
    # ----------------------------------------------------------
    result_df = df.drop(*columns_to_remove) if columns_to_remove else df

    print("\n" + "=" * 70)
    print(f"Colunas originais : {len(df.columns):,}")
    print(f"Colunas removidas : {len(columns_to_remove):,}")
    print(f"Colunas finais    : {len(result_df.columns):,}")
    print("=" * 70)

    return result_df, columns_to_remove


In [0]:
abt_00, removed_cols = remove_high_missing_columns_sampled(df_temp_01, threshold=70)


print(f"\nDataFrame processado:")
print(f"Linhas: {abt_00.count():,}")
print(f"Colunas: {len(abt_00.columns):,}")